In [3]:
import time
from typing import TypedDict, Annotated
from operator import add
from langgraph.cache.memory import InMemoryCache
from langgraph.graph import StateGraph,START,END
from langgraph.types import CachePolicy
from loguru import  logger

class OverAllState(TypedDict):
    user:str
    invoke_counts:Annotated[int,add]
def node_a(state:OverAllState) ->OverAllState:
    logger.info("node_a被调用 user:{}",state["user"])
    time.sleep(3) #模拟耗时操作
    logger.info("node_a耗时操作执行完成")
    return {
        "invoke_counts":1
    }

builder=StateGraph(state_schema=OverAllState)
builder.add_node(
    "node_a",
    node_a,
    cache_policy=CachePolicy(ttl = 20)
)

builder.add_edge(START,"node_a")
builder.add_edge("node_a",END)

graph=builder.compile(cache= InMemoryCache())
logger.info("首次调用图")
logger.info("运行结果 {}\n\n",graph.invoke({"user":"小明","invoke_counts":0}))
logger.info("相同输入再次调用图")
logger.info("运行结果 {}\n\n",graph.invoke({"user":"小明","invoke_counts":0}))
logger.info("不同的输入调用图")
logger.info("运行结果 {}\n\n",graph.invoke({"user":"小花","invoke_counts":0}))
logger.info("不同的输入再次调用图")
logger.info("运行结果 {}\n\n",graph.invoke({"user":"小花","invoke_counts":0}))

#缓存超时
time.sleep(11)
logger.info("不同的输入再次调用图")
logger.info("运行结果 {}\n\n",graph.invoke({"user":"小花","invoke_counts":0}))

2026-09-25 15:56:28.115 | INFO     | __main__:<module>:31 - 首次调用图
2026-09-25 15:56:28.117 | INFO     | __main__:node_a:13 - node_a被调用 user:小明
2026-09-25 15:56:31.118 | INFO     | __main__:node_a:15 - node_a耗时操作执行完成
2026-09-25 15:56:31.121 | INFO     | __main__:<module>:32 - 运行结果 {'user': '小明', 'invoke_counts': 1}


2026-09-25 15:56:31.122 | INFO     | __main__:<module>:33 - 相同输入再次调用图
2026-09-25 15:56:31.125 | INFO     | __main__:<module>:34 - 运行结果 {'user': '小明', 'invoke_counts': 1}


2026-09-25 15:56:31.127 | INFO     | __main__:<module>:35 - 不同的输入调用图
2026-09-25 15:56:31.130 | INFO     | __main__:node_a:13 - node_a被调用 user:小花
2026-09-25 15:56:34.131 | INFO     | __main__:node_a:15 - node_a耗时操作执行完成
2026-09-25 15:56:34.134 | INFO     | __main__:<module>:36 - 运行结果 {'user': '小花', 'invoke_counts': 1}


2026-09-25 15:56:34.134 | INFO     | __main__:<module>:37 - 不同的输入再次调用图
2026-09-25 15:56:34.135 | INFO     | __main__:<module>:38 - 运行结果 {'user': '小花', 'invoke_counts': 1}


2026-09-25 15:56:4